# 🔬 Facial Erythema & Redness Rash Segmentation (YOLO11s-seg)
Automated instance segmentation training pipeline for facial erythema, rosacea, and skin redness rashes.

### Prerequisites:
1. Change runtime to GPU: **Runtime -> Change runtime type -> T4 GPU**.
2. Upload your dataset zip archive into the left panel (📁) or use Roboflow API.
3. Click **Runtime -> Run all** (`Ctrl + F9`).

In [ ]:
# 1. Verify GPU availability
!nvidia-smi

In [ ]:
# 2. Install required packages
!pip install -q ultralytics roboflow pyyaml

In [ ]:
# 3. Dataset Setup
import os, shutil, glob, yaml

dataset_dir = '/content/dataset'
if os.path.exists(dataset_dir):
    shutil.rmtree(dataset_dir)

for split in ['train', 'val']:
    os.makedirs(f'{dataset_dir}/images/{split}', exist_ok=True)
    os.makedirs(f'{dataset_dir}/labels/{split}', exist_ok=True)

# Check for uploaded zip archive
zips = glob.glob('/content/*.zip')
if zips:
    zip_path = zips[0]
    print(f'==> Unpacking uploaded dataset: {zip_path}')
    !unzip -q -o {zip_path} -d /content/dataset
else:
    print('==> No local zip found. Enter your Roboflow API Key to download datasets:')
    from google.colab import userdata
    try:
        api_key = userdata.get('ROBOFLOW_API_KEY')
    except Exception:
        api_key = input('Roboflow API Key: ').strip()
    
    if api_key:
        from roboflow import Roboflow
        rf = Roboflow(api_key=api_key)
        ds1 = rf.workspace('harsh-iqsff').project('redness-pdald').version(1).download('yolov11')
        ds2 = rf.workspace('blowproject').project('redness-0ov9b-njenr').version(1).download('yolov11')
        
        for ds in [ds1, ds2]:
            loc = ds.location
            for s_from, s_to in [('train', 'train'), ('valid', 'val'), ('test', 'val')]:
                img_in = os.path.join(loc, s_from, 'images')
                lbl_in = os.path.join(loc, s_from, 'labels')
                if not os.path.exists(img_in): continue
                for fn in os.listdir(img_in):
                    if not fn.lower().endswith(('.jpg', '.png', '.jpeg')): continue
                    bn, _ = os.path.splitext(fn)
                    lf = os.path.join(lbl_in, bn + '.txt')
                    if os.path.exists(lf):
                        shutil.copy2(os.path.join(img_in, fn), f'{dataset_dir}/images/{s_to}/{fn}')
                        shutil.copy2(lf, f'{dataset_dir}/labels/{s_to}/{bn}.txt')

data_yaml = {
    'path': '/content/dataset',
    'train': 'images/train',
    'val': 'images/val',
    'nc': 1,
    'names': ['redness_rash']
}
with open('/content/dataset/data.yaml', 'w') as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print('Dataset ready! Training images:', len(os.listdir(f'{dataset_dir}/images/train')))

In [ ]:
# 4. Training YOLO11s-seg
from ultralytics import YOLO

model = YOLO('yolo11s-seg.pt')

results = model.train(
    data='/content/dataset/data.yaml',
    epochs=80,
    imgsz=640,
    batch=16,
    cache='ram',
    workers=2,
    device=0,
    project='/content/runs',
    name='erythema_colab_run',
    exist_ok=True,
    mosaic=0.5,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.4,
    degrees=10.0,
    fliplr=0.5,
    save=True
)

In [ ]:
# 5. Save & Download best.pt
import os, shutil
from google.colab import files

best_pt = '/content/runs/erythema_colab_run/weights/best.pt'
if os.path.exists(best_pt):
    print('🎉 Training complete!')
    if os.path.exists('/content/drive/MyDrive'):
        shutil.copy2(best_pt, '/content/drive/MyDrive/erythema_best.pt')
        print('Saved to Google Drive: /content/drive/MyDrive/erythema_best.pt')
    files.download(best_pt)
else:
    print('Weights file not found.')